# Experiment 9: Adagrad Optimizer in Deep Learning

**Objective**: Understand and demonstrate the Adagrad (Adaptive Gradient) optimizer, analyze how it adapts learning rates per-parameter, and compare its performance against other popular optimizers.

**Dataset**: MNIST (Handwritten Digits)

**Methodology**:
1. Load and preprocess the MNIST dataset.
2. Define a neural network architecture.
3. Train the model using Adagrad with different learning rates.
4. Compare Adagrad with SGD, RMSprop, and Adam optimizers.
5. Visualize training curves and analyze convergence behavior.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")

## 2. Understanding Adagrad

**Adagrad (Adaptive Gradient Algorithm)** adapts the learning rate for each parameter individually based on the history of gradients.

**Update Rule:**

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \epsilon}} \cdot g_t$$

Where:
- $\theta_t$ = parameter at time step $t$
- $\eta$ = initial learning rate
- $G_t$ = sum of squares of all past gradients for that parameter
- $g_t$ = gradient at time step $t$
- $\epsilon$ = small constant to avoid division by zero

**Key Properties:**
- Parameters with large gradients get smaller effective learning rates.
- Parameters with small gradients get larger effective learning rates.
- Well-suited for sparse data (e.g., NLP tasks).
- **Drawback**: The accumulated squared gradients $G_t$ grow monotonically, causing the effective learning rate to shrink to near zero over time.

## 3. Load and Preprocess Dataset

In [ ]:
# Load MNIST dataset
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to be between 0 and 1
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Flatten the images (28x28 -> 784)
X_train = X_train.reshape((-1, 784))
X_test = X_test.reshape((-1, 784))

# One-hot encode the labels
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

## 4. Define Model Architecture

In [ ]:
def create_model(optimizer):
    """
    Creates a simple MLP model compiled with the given optimizer.
    
    Parameters:
    -----------
    optimizer : keras.optimizers.Optimizer
        The optimizer instance to use for training.
    """
    model = keras.Sequential([
        layers.Dense(512, activation='relu', input_shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## 5. Experiment A: Adagrad with Different Learning Rates

We train the same model architecture using Adagrad with varying initial learning rates to see how the learning rate affects convergence.

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.1]
adagrad_histories = {}

epochs = 20
batch_size = 128

for lr in learning_rates:
    print(f"\n{'='*50}")
    print(f"Training Adagrad with Learning Rate: {lr}")
    print(f"{'='*50}")
    
    optimizer = keras.optimizers.Adagrad(learning_rate=lr)
    model = create_model(optimizer)
    
    history = model.fit(X_train, y_train,
                        epochs=epochs,
                        batch_size=batch_size,
                        validation_split=0.2,
                        verbose=1)
    
    # Evaluate on test set
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Accuracy: {test_acc:.4f}  |  Test Loss: {test_loss:.4f}")
    
    adagrad_histories[lr] = history.history
    adagrad_histories[lr]['test_accuracy'] = test_acc
    adagrad_histories[lr]['test_loss'] = test_loss

## 6. Visualize Adagrad Learning Rate Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Validation Accuracy
for lr in learning_rates:
    axes[0].plot(adagrad_histories[lr]['val_accuracy'], label=f'LR={lr}')
axes[0].set_title('Adagrad: Validation Accuracy vs Epochs', fontsize=14)
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Validation Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation Loss
for lr in learning_rates:
    axes[1].plot(adagrad_histories[lr]['val_loss'], label=f'LR={lr}')
axes[1].set_title('Adagrad: Validation Loss vs Epochs', fontsize=14)
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Validation Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart of final test accuracy for each LR
test_accs = [adagrad_histories[lr]['test_accuracy'] for lr in learning_rates]
labels = [f'LR={lr}' for lr in learning_rates]

plt.figure(figsize=(10, 5))
bars = plt.bar(labels, test_accs, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
plt.title('Adagrad: Test Accuracy for Different Learning Rates', fontsize=14)
plt.ylabel('Test Accuracy')
plt.ylim(min(test_accs) - 0.02, max(test_accs) + 0.01)

# Add value labels on bars
for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=12)

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Experiment B: Compare Adagrad with Other Optimizers

We compare Adagrad against SGD, RMSprop, and Adam to understand relative strengths and weaknesses.

In [ ]:
optimizers = {
    'SGD':     keras.optimizers.SGD(learning_rate=0.01),
    'Adagrad': keras.optimizers.Adagrad(learning_rate=0.01),
    'RMSprop': keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam':    keras.optimizers.Adam(learning_rate=0.001)
}

optimizer_histories = {}

for name, opt in optimizers.items():
    print(f"\n{'='*50}")
    print(f"Training with {name} Optimizer")
    print(f"{'='*50}")
    
    model = create_model(opt)
    
    history = model.fit(X_train, y_train,
                        epochs=epochs,
                        batch_size=batch_size,
                        validation_split=0.2,
                        verbose=1)
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Test Accuracy: {test_acc:.4f}  |  Test Loss: {test_loss:.4f}")
    
    optimizer_histories[name] = history.history
    optimizer_histories[name]['test_accuracy'] = test_acc
    optimizer_histories[name]['test_loss'] = test_loss

## 8. Visualize Optimizer Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

colors = {'SGD': '#e74c3c', 'Adagrad': '#2ecc71', 'RMSprop': '#3498db', 'Adam': '#f39c12'}

# Training Accuracy
for name in optimizers:
    axes[0, 0].plot(optimizer_histories[name]['accuracy'], label=name, color=colors[name], linewidth=2)
axes[0, 0].set_title('Training Accuracy vs Epochs', fontsize=13)
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Validation Accuracy
for name in optimizers:
    axes[0, 1].plot(optimizer_histories[name]['val_accuracy'], label=name, color=colors[name], linewidth=2)
axes[0, 1].set_title('Validation Accuracy vs Epochs', fontsize=13)
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Training Loss
for name in optimizers:
    axes[1, 0].plot(optimizer_histories[name]['loss'], label=name, color=colors[name], linewidth=2)
axes[1, 0].set_title('Training Loss vs Epochs', fontsize=13)
axes[1, 0].set_xlabel('Epochs')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Validation Loss
for name in optimizers:
    axes[1, 1].plot(optimizer_histories[name]['val_loss'], label=name, color=colors[name], linewidth=2)
axes[1, 1].set_title('Validation Loss vs Epochs', fontsize=13)
axes[1, 1].set_xlabel('Epochs')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Optimizer Comparison: SGD vs Adagrad vs RMSprop vs Adam', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison of final test accuracy
opt_names = list(optimizers.keys())
opt_test_accs = [optimizer_histories[name]['test_accuracy'] for name in opt_names]
opt_test_losses = [optimizer_histories[name]['test_loss'] for name in opt_names]
bar_colors = [colors[name] for name in opt_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test Accuracy
bars1 = axes[0].bar(opt_names, opt_test_accs, color=bar_colors)
axes[0].set_title('Test Accuracy by Optimizer', fontsize=13)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(min(opt_test_accs) - 0.03, max(opt_test_accs) + 0.015)
for bar, acc in zip(bars1, opt_test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{acc:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Test Loss
bars2 = axes[1].bar(opt_names, opt_test_losses, color=bar_colors)
axes[1].set_title('Test Loss by Optimizer', fontsize=13)
axes[1].set_ylabel('Loss')
for bar, loss in zip(bars2, opt_test_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{loss:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Convergence Speed Analysis

In [ ]:
# Find epochs required to reach 95% and 97% validation accuracy for each optimizer
thresholds = [0.95, 0.97]

print(f"{'Optimizer':<12} | {'Epochs to 95%':>15} | {'Epochs to 97%':>15}")
print('-' * 50)

for name in optimizers:
    val_accs = optimizer_histories[name]['val_accuracy']
    results = []
    for thresh in thresholds:
        epoch_reached = None
        for i, acc in enumerate(val_accs):
            if acc >= thresh:
                epoch_reached = i + 1
                break
        results.append(str(epoch_reached) if epoch_reached else 'Not reached')
    print(f"{name:<12} | {results[0]:>15} | {results[1]:>15}")

## 10. Summary Table

In [ ]:
import pandas as pd

summary_data = []
for name in optimizers:
    h = optimizer_histories[name]
    summary_data.append({
        'Optimizer': name,
        'Final Train Accuracy': f"{h['accuracy'][-1]:.4f}",
        'Final Val Accuracy': f"{h['val_accuracy'][-1]:.4f}",
        'Final Train Loss': f"{h['loss'][-1]:.4f}",
        'Final Val Loss': f"{h['val_loss'][-1]:.4f}",
        'Test Accuracy': f"{h['test_accuracy']:.4f}",
        'Test Loss': f"{h['test_loss']:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
summary_df

## Conclusion / Observations

1. **Adagrad's Adaptive Learning Rate**: Adagrad automatically scales down the learning rate for parameters that receive frequent, large gradient updates. This makes it effective for problems with sparse features.

2. **Effect of Initial Learning Rate on Adagrad**: A very low initial learning rate (e.g., 0.001) may cause Adagrad to converge too slowly because the effective rate shrinks further. A moderate rate (0.01–0.05) generally works best.

3. **Adagrad vs SGD**: Adagrad typically converges faster than vanilla SGD (especially without momentum) because it adapts per-parameter learning rates, avoiding the need for careful manual tuning.

4. **Adagrad vs Adam/RMSprop**: Adam and RMSprop address Adagrad's main weakness — the monotonically decreasing learning rate — by using exponential moving averages of past squared gradients instead of the full sum. This generally leads to better performance on longer training runs.

5. **When to Use Adagrad**:
   - Sparse data scenarios (e.g., NLP, recommendation systems).
   - When you need per-parameter adaptive learning rates but the training run is short.
   - When manual learning rate scheduling is undesirable.

6. **Adagrad's Limitation**: The accumulated gradient sum $G_t$ grows without bound, causing the effective learning rate to decay to near-zero. This means training can effectively stall in later epochs — a problem solved by RMSprop and Adam.